**import's**

In [1]:
import pandas as pd
import numpy as np
import random
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import  roc_auc_score

**config**

In [2]:
NUM_RANGE = 49
N_SPLITS = 5
RANDOM_STATE = 42

**load  data**

In [3]:
def load_data(path):
  df= pd.read_csv(path, parse_dates=['Date'])
  df.sort_values('Date').reset_index(drop=True)
  return df

from google.colab import drive
drive.mount('/content/drive')

df = load_data('/content/drive/MyDrive/Colab Notebooks/sg_pools/ToTo.csv')
display(df.head(2))

Mounted at /content/drive


,Draw,Date,Winning Number 1,2,3,4,5,6,Additional Number,From Last,...,Division 3 Winners,Division 3 Prize,Division 4 Winners,Division 4 Prize,Division 5 Winners,Division 5 Prize,Division 6 Winners,Division 6 Prize,Division 7 Winners,Division 7 Prize
0,4178,2026-04-30,2,6,7,31,35,39,15,NaN,...,223.0,1818.0,639.0,346.0,12572.0,50.0,18347.0,25.0,238219.0,10.0
1,4177,2026-04-27,3,11,13,22,28,48,21,3,...,163.0,1522.0,420.0,323.0,8770.0,50.0,11113.0,25.0,159071.0,10.0


**transform**

In [4]:
def create_binary_matrix(df):
  records = []

  for _, row in df.iterrows():
    nums = set(row[['Winning Number 1','2','3','4','5','6']])
    record = {f"num_{i}": (1 if i in nums else 0) for i in range(1, NUM_RANGE+1)}
    record['Date'] = row['Date']
    records.append(record)
  return pd.DataFrame( records)

**feature engineering**

In [5]:
def add_features(bin_df):
  df = bin_df.copy()

  for i in range(1, NUM_RANGE+1):
    col = f"num_{i}"

    # rolling frequencies
    df[f"{col}_freq_10"] = df[col].rolling(10).mean()
    df[f"{col}_freq_30"] = df[col].rolling(30).mean()

    # lag
    df[f"{col}_lag1"] = df[col].shift(1)

    # gap
    gap, last_seen = [], -1
    for idx, val in enumerate(df[col]):
      if val == 1:
        gap.append(0)
        last_seen = idx
      else:
        gap.append(idx - last_seen if last_seen != -1 else -1)
    df[f"{col}_gap"] = gap

  df = df.dropna().reset_index(drop=True)
  return df

add_features(create_binary_matrix(df))

/tmp/ipykernel_2061/1995129024.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag1"] = df[col].shift(1)
/tmp/ipykernel_2061/1995129024.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_gap"] = gap
/tmp/ipykernel_2061/1995129024.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.

,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,num_10,...,num_47_lag1,num_47_gap,num_48_freq_10,num_48_freq_30,num_48_lag1,num_48_gap,num_49_freq_10,num_49_freq_30,num_49_lag1,num_49_gap
0,0,0,0,1,0,0,0,0,0,0,...,0.0,4,0.3,0.233333,0.0,3,0.2,0.166667,0.0,6
1,0,0,0,0,0,0,0,0,0,0,...,0.0,5,0.2,0.233333,0.0,4,0.2,0.166667,0.0,7
2,1,0,0,0,0,0,0,0,1,0,...,0.0,6,0.2,0.200000,0.0,5,0.2,0.166667,0.0,8
3,0,0,1,0,0,0,0,0,0,0,...,0.0,7,0.2,0.200000,0.0,6,0.1,0.166667,0.0,9
4,0,0,0,0,1,0,0,0,0,0,...,0.0,8,0.2,0.200000,0.0,7,0.1,0.166667,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1804,0,0,0,0,1,0,0,0,0,1,...,0.0,650,0.0,0.000000,0.0,651,0.0,0.000000,0.0,653
1805,0,0,0,0,1,0,0,0,0,0,...,0.0,651,0.0,0.000000,0.0,652,0.0,0.000000,0.0,654
1806,0,0,0,0,0,0,0,0,0,0,...,0.0,652,0.0,0.000000,0.0,653,0.0,0.000000,0.0,655
1807,0,1,0,0,0,0,0,0,0,0,...,0.0,653,0.0,0.000000,0.0,654,0.0,0.000000,0.0,656


**training models**

In [13]:
def train_models(feat_df):
  models = {}
  scores = {}

  tscv = TimeSeriesSplit(n_splits=N_SPLITS)
  # Corrected feature_cols definition to include only engineered features
  feature_cols = [c for c in feat_df.columns if "_freq" in c or "_lag" in c or "_gap" in c]

  for i in range(1, NUM_RANGE+1):
    target = f"num_{i}"
    x = feat_df[feature_cols]
    y = feat_df[target]

    model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1)
    fold_scores = []
    for train_idx, val_idx in tscv.split(x):
      x_train, x_val = x.iloc[train_idx], x.iloc[val_idx]
      y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
      model.fit(x_train, y_train)
      y_pred = model.predict_proba(x_val)[:, 1]
      try:
        fold_scores.append(roc_auc_score(y_val, y_pred))
      except: pass

      models[i] = model
      scores[i] = np.mean(fold_scores) if fold_scores else 0.5

  print("average AUC:", np.mean(list(scores.values())))
  return models

**predict probability**

In [7]:
def predict_probs(models, latest_row):
  probs = {}
  for i in range(1, NUM_RANGE+1):
    model = models[i]
    probs[i] = model.predict_proba(latest_row)[0][1]
  return probs

**monte carlo**

In [8]:
def monte_carlo_population(probs, size=1000):
  numbers = list(prob.keys())
  weights = np.array(list(probs.values()))
  weights = weights / weights.sum()
  population = set()

  while len(population) < size:
    combo = tuple(sorted(np.random.choice(numbers, 6, replace=False, p=weights)))
    population.add(combo)

  return [list(c) for c in population]

def compute_mc_frequency(population):
  counter = Counter(tuple(sorted(p)) for p in population)
  total = sum(counter.values())
  return {k:v/total for k, v in counter.items()}

**ga component**

In [9]:
def fitness(combo,  probs, mc_freq):
  combo_tuple = tuple(combo)
  prob_score = sum(probs[n]  for n  in  combo)
  mc_score = mc_freq.get(combo_tuple, 0)
  consecutive_penalty = sum(1 for i in range(len(combo)-1) if combo[i]+1 == combo[i+1])
  s = sum(combo)
  sum_penalty = 0 if 90 <= s <= 180 else 1
  odd_count = sum(n%2 for n in combo)
  balance_penalty = abs(odd_count - 3)
  return prob_score * 0.5 + mc_score * 0.3 - consecutive_penalty * 0.1 - sum_penalty * 0.05 - balance_penalty * 0.05

def tournament_selection(population, scores, k=3):
  selected = random.sample(list(zip(population, scores)), k)
  selected.sort(key=lambda x: x[1], reverse=True)
  return selected[0][0]

def crossover(p1, p2):
  child = list(set(p1[:3] + p2[3:]))
  while len(child) < 6:
    n = random.randint(1, 49)
    if n not in child:
      child.append(n)
  return sorted(child)

def mutate(ind, rate=0.2):
  if random.random() < rate:
    idx = random.randint(0, 5)
    new_val = random.randint(1, 49)
    while new_val  in ind:
      new_val = random.randint(1, 49)
    ind[idx] = new_val
  return sorted(ind)


**hybrid ga**

In [10]:
def run_hybrid_ga(probs, generation=50, pop_size=300):
  population = monte_carlo_population(probs, size=pop_size)
  mc_freq = compute_mc_frequency(population)
  for gen in range(generation):
    scores = [fitness(ind, probs, mc_freq) for ind in population]
    ranked = sorted(zip(population, scores), key=lambda x: x[1], reverse=True)
    elites = [r[0] for r in ranked[:20]]
    new_population = elites.copy()
    while len(new_population) < pop_size:
      p1 = tournament_selection(population, scores)
      p2 = tournament_selection(population, scores)
      child = crossover(p1, p2)
      child = mutate(child)
      new_population.append(child)
    population = new_population
    if gen % 10 == 0:
      print(f"Gen {gen} Best Score: {ranked[0][1]:.4f}")
  final = [(ind, fitness(ind, probs, mc_freq)) for ind in population]
  final.sort(key=lambda x: x[1], reverse=True)
  return final[:10]


**backtest (real validation)**

In [14]:
def backtest(models, feat_df):
  hits = []
  # Corrected feature_cols definition to include only engineered features
  feature_cols = [c for c in feat_df.columns if "_freq" in c or "_lag" in c or "_gap" in c]
  for i in range(len(feat_df)-1):
    X = feat_df.iloc[[i]][feature_cols]
    actual = feat_df.iloc[i+1]
    probs = predict_probs(models, X)
    pred = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:6]
    pred_nums = set(n for n,_ in pred)
    actual_nums = set(j for j in range(1, 50) if actual[f'num_{j}'] == 1)
    hits.append(len(pred_nums & actual_nums))
    print("Avg hits:", np.mean(hits))

**main**

In [ ]:
def main():
  df = load_data('/content/drive/MyDrive/Colab Notebooks/sg_pools/ToTo.csv')
  bin_df = create_binary_matrix(df)
  feat_df = add_features(bin_df)
  models = train_models(feat_df)

  # backtest
  backtest(models, feat_df)

  # latest prediction
  # Corrected feature_cols definition to include only engineered features
  feature_cols = [c for c in feat_df.columns if "_freq" in c or "_lag" in c or "_gap" in c]
  # Corrected typo: ilocs to iloc
  latest = feat_df.iloc[[-1]][feature_cols]

  probs = predict_probs(models, latest)

  # hybrid optimization
  results = run_hybrid_ga(probs)

  print("\nTop combinations:")
  for combo, score in results:
    print(combo,score)

if __name__ == "__main__":
  main()

/tmp/ipykernel_2061/1995129024.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_lag1"] = df[col].shift(1)
/tmp/ipykernel_2061/1995129024.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_gap"] = gap
/tmp/ipykernel_2061/1995129024.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.

average AUC: nan
Avg hits: 1.0
Avg hits: 1.5
Avg hits: 1.0
Avg hits: 0.75
Avg hits: 1.0
Avg hits: 0.8333333333333334
Avg hits: 0.7142857142857143
Avg hits: 0.625
Avg hits: 0.6666666666666666
Avg hits: 0.6
Avg hits: 0.5454545454545454
Avg hits: 0.5
Avg hits: 0.46153846153846156
Avg hits: 0.5
Avg hits: 0.4666666666666667
Avg hits: 0.5
Avg hits: 0.5294117647058824
Avg hits: 0.5
Avg hits: 0.5789473684210527
Avg hits: 0.55
Avg hits: 0.5238095238095238
Avg hits: 0.5
Avg hits: 0.5217391304347826
Avg hits: 0.5416666666666666
Avg hits: 0.6
Avg hits: 0.5769230769230769
Avg hits: 0.5555555555555556
Avg hits: 0.5357142857142857
Avg hits: 0.5517241379310345
Avg hits: 0.6333333333333333
Avg hits: 0.6451612903225806
Avg hits: 0.65625
Avg hits: 0.6363636363636364
Avg hits: 0.6176470588235294
Avg hits: 0.6285714285714286
Avg hits: 0.6111111111111112
Avg hits: 0.6216216216216216
Avg hits: 0.6052631578947368
Avg hits: 0.6153846153846154
Avg hits: 0.625
Avg hits: 0.6097560975609756
Avg hits: 0.61904761904